# 🐝 Heady Brain — GPU Runtime

**Chat directly with HeadyBrain. Keys load automatically — just run the cells.**

1. Run Cell 1 (loads keys from Heady's memory)
2. Run Cell 2 (chat engine with **persistent conversation history**)
3. Run Cell 3 to talk!

### Conversation Commands
- `chat("msg")` — continues the current conversation
- `new_chat()` — start a fresh conversation
- `save_chat("name")` — save current conversation to disk
- `load_chat("name")` — load a saved conversation
- `show_history()` — view conversation so far

In [ ]:
# ═══ Cell 1: Load Keys from Heady Memory ═══
import base64 as _b, os

def _d(s): return _b.b64decode(s).decode()

# Heady's key vault — loaded from memory
_vault = {
    'gemini': [
        (_d('QUl6YVN5RE9MNVlKX042Q0xlVk5ReXRCeURENm5aMEktS3I0WVJr'), 'HC-Heady'),
        (_d('QUl6YVN5QllSWEtTckdDVXBfWUxhbGo5R0NRNnRtbjlNZENpVXNn'), 'HC-GCloud'),
        (_d('QUl6YVN5Q3hkTlowQUpFVVoxUi1iQWljaDJIa28tRHpSb0w4RjVv'), 'HM-2'),
        (_d('QUl6YVN5RENZTXJjYkVONTF1d0lteXR2RUtPeUVFbXVaZGc4TFBr'), 'HM-3'),
        (_d('QUl6YVN5QkxUdTBoOVEwOUNyMDVfM19aal8zeWVudDVjTzNpYUhF'), 'HM-4'),
        (_d('QUl6YVN5REIxU2MyUHpzUld4TldvaGNJQ0dIQXFRS001WXZSY1hv'), 'HM-Default'),
    ],
    'openai': [
        (_d('c2stcHJvai1mb29fcDRGa2JCWklDTENRcFc4T3VmMlRLTlNZVlBrTnd5Q3lKUlBDQ29xVmFITWVLT3NKSF9Ha0h5bHhKR1JzZW85VHQySVo3QlQzQmxia0ZKT2ZXNThjTTh6LUFDMXY5dUt4YjdkelpoMjNLXy1XM0lmb0U5NkxCME44NXJlVEp2V1Mxa0llOHFxM2ZkTzlsdEVVUHJWOUNWVUE='), 'Seat1'),
        (_d('c2stc3ZjYWNjdC0xNzB6NFhLYnI4dE0xTDBRbkpsMEMtS0lOeldaemFLM0E3RVpIaDJMRjNHczJqUUplZEpuallJRUFHb3A2OUwxbGY1WGwyajcyS1QzQmxia0ZKdTNHRnFZUVZxdDVuc2ZxdmlFNm1xUXVyUmlYM19DbXk1TVYxRjlFdUVzSU1odHNZaTVyNmZ2YURnWGRYWWN4VGZ0cUZWZXctd0E='), 'SvcAcct'),
    ],
    'anthropic': [
        (_d('c2stYW50LWFwaTAzLXN4cnZjRkVmdzBZMkJVWnVySVNidmhaS0N0OWg3bTNyRDZXa3hUaDJKekR4U2N0MlFENjZTZTEybjRjbkIxUVM1Nk1IYWNQWFgwVERXUG8wM0ppSU1RLWRRYkJid0FB'), 'Claude-API03'),
    ],
    'huggingface': [
        (_d('aGZfdW1IdXlLV1pVU2tXaG54U3RaQVFORUdtSFNXenBlU0dLaw=='), 'HF-Seat1'),
        (_d('aGZfT3lwU0JwSVlNaERrR3BEVkZoYVZJSUxPRmVnUlpWV3hPdA=='), 'HF-Seat2'),
    ],
    'perplexity': [],
}

# Also check Colab Secrets and env vars as overrides
def _get(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except: pass
    return os.environ.get(name)

for name, provider in [('GEMINI_API_KEY','gemini'),('OPENAI_API_KEY','openai'),('ANTHROPIC_API_KEY','anthropic'),('HF_TOKEN','huggingface'),('PERPLEXITY_API_KEY','perplexity')]:
    v = _get(name)
    if v and not any(k==v for k,_ in _vault.get(provider,[])): _vault[provider].append((v, f'env:{name}'))

KEYS = _vault
total = sum(len(v) for v in KEYS.values())
print(f'🐝 Heady Memory: {total} keys loaded automatically')
for p, ks in KEYS.items():
    if ks: print(f'  ✅ {p}: {len(ks)} key(s) [{', '.join(l for _,l in ks)}]')
    else:  print(f'  ⬜ {p}: none')

In [ ]:
# ═══ Cell 2: Chat Engine with Persistent Conversation History ═══
import json, urllib.request, time, concurrent.futures, os, uuid

SYSTEM = '''You are HeadyBrain, the sovereign AI reasoning engine of the Heady ecosystem.
3 accounts (HeadySystems, HeadyConnection, HeadyMe), 5 domains, HuggingFace Spaces, Cloud Run.
Be helpful, precise, and actionable.'''

# ─── Persistent Conversation History ───────────────────────────────
# Every message you send and every response is accumulated here.
# The FULL history is sent with each request so the AI has context.
conversation_history = []  # [{"role": "user"/"assistant", "content": "..."}]
session_id = str(uuid.uuid4())[:8]
MAX_HISTORY = 50  # sliding window — keep last 50 messages
SESSIONS_DIR = '/content/heady_sessions'  # Colab persistent dir

def _trim_history():
    global conversation_history
    if len(conversation_history) > MAX_HISTORY:
        conversation_history = conversation_history[-MAX_HISTORY:]

def new_chat():
    """Start a fresh conversation. Clears all history."""
    global conversation_history, session_id
    conversation_history = []
    session_id = str(uuid.uuid4())[:8]
    print(f'🆕 New conversation started (session: {session_id})')

def show_history():
    """Show the current conversation history."""
    if not conversation_history:
        print('📭 No conversation history yet.')
        return
    print(f'📜 Conversation ({len(conversation_history)} messages, session: {session_id}):\n')
    for i, msg in enumerate(conversation_history):
        role = '👤 You' if msg['role'] == 'user' else '🐝 Heady'
        preview = msg['content'][:200] + ('...' if len(msg['content']) > 200 else '')
        print(f'  {i+1}. {role}: {preview}\n')

def save_chat(name='auto'):
    """Save current conversation to disk."""
    os.makedirs(SESSIONS_DIR, exist_ok=True)
    path = os.path.join(SESSIONS_DIR, f'{name}.json')
    data = {'session_id': session_id, 'history': conversation_history, 'saved_at': time.strftime('%Y-%m-%dT%H:%M:%S')}
    with open(path, 'w') as f: json.dump(data, f, indent=2)
    print(f'💾 Saved {len(conversation_history)} messages to {path}')

def load_chat(name='auto'):
    """Load a saved conversation from disk."""
    global conversation_history, session_id
    path = os.path.join(SESSIONS_DIR, f'{name}.json')
    if not os.path.exists(path):
        print(f'❌ No saved chat \"{name}\" found at {path}')
        return
    with open(path) as f: data = json.load(f)
    conversation_history = data.get('history', [])
    session_id = data.get('session_id', name)
    print(f'📂 Loaded {len(conversation_history)} messages (session: {session_id}, saved: {data.get(\"saved_at\", \"unknown\")})')

def list_chats():
    """List all saved conversations."""
    if not os.path.exists(SESSIONS_DIR):
        print('📭 No saved chats yet.')
        return
    files = [f for f in os.listdir(SESSIONS_DIR) if f.endswith('.json')]
    if not files:
        print('📭 No saved chats yet.')
        return
    print(f'📚 Saved chats ({len(files)}):')
    for f in sorted(files):
        path = os.path.join(SESSIONS_DIR, f)
        try:
            with open(path) as fh: d = json.load(fh)
            print(f'  📄 {f[:-5]} — {len(d.get("history",[]))} msgs, saved {d.get("saved_at","?")}')
        except: print(f'  ⚠ {f} (corrupt)')


# ─── Provider Call with Full History ─────────────────────────────
def _call(provider, key, message):
    """Call a provider with the FULL conversation history for context."""
    if provider == 'gemini':
        # Gemini uses 'contents' array with 'user'/'model' roles
        contents = []
        for msg in conversation_history:
            role = 'model' if msg['role'] == 'assistant' else 'user'
            contents.append({'role': role, 'parts': [{'text': msg['content']}]})
        contents.append({'role': 'user', 'parts': [{'text': message}]})
        url = f'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={key}'
        data = json.dumps({
            'systemInstruction': {'parts': [{'text': SYSTEM}]},
            'contents': contents,
            'generationConfig': {'maxOutputTokens': 8192}
        }).encode()
        req = urllib.request.Request(url, data=data, headers={'Content-Type': 'application/json'})
        d = json.loads(urllib.request.urlopen(req, timeout=120).read())
        return d['candidates'][0]['content']['parts'][0]['text']

    elif provider == 'openai':
        # OpenAI uses standard messages array
        messages = [{'role': 'system', 'content': SYSTEM}]
        messages.extend(conversation_history)
        messages.append({'role': 'user', 'content': message})
        data = json.dumps({'model': 'gpt-4o', 'messages': messages, 'max_tokens': 8192}).encode()
        req = urllib.request.Request('https://api.openai.com/v1/chat/completions', data=data, headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {key}'})
        d = json.loads(urllib.request.urlopen(req, timeout=120).read())
        return d['choices'][0]['message']['content']

    elif provider == 'anthropic':
        # Anthropic uses messages array (no system in messages)
        messages = list(conversation_history)
        messages.append({'role': 'user', 'content': message})
        data = json.dumps({'model': 'claude-sonnet-4-20250514', 'max_tokens': 8192, 'system': SYSTEM, 'messages': messages}).encode()
        req = urllib.request.Request('https://api.anthropic.com/v1/messages', data=data, headers={'Content-Type': 'application/json', 'x-api-key': key, 'anthropic-version': '2023-06-01'})
        d = json.loads(urllib.request.urlopen(req, timeout=120).read())
        return d['content'][0]['text']

    elif provider == 'perplexity':
        messages = [{'role': 'system', 'content': SYSTEM}]
        messages.extend(conversation_history)
        messages.append({'role': 'user', 'content': message})
        data = json.dumps({'model': 'sonar-pro', 'messages': messages, 'max_tokens': 4096}).encode()
        req = urllib.request.Request('https://api.perplexity.ai/chat/completions', data=data, headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {key}'})
        d = json.loads(urllib.request.urlopen(req, timeout=120).read())
        return d['choices'][0]['message']['content']

    elif provider == 'huggingface':
        messages = list(conversation_history)
        messages.append({'role': 'user', 'content': message})
        data = json.dumps({'model': 'deepseek/deepseek-r1-0528', 'messages': messages, 'max_tokens': 4096, 'stream': False}).encode()
        req = urllib.request.Request('https://router.huggingface.co/novita/v3/openai/chat/completions', data=data, headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {key}'})
        d = json.loads(urllib.request.urlopen(req, timeout=120).read())
        return d['choices'][0]['message']['content']


def chat(message, provider='auto'):
    """Chat with HeadyBrain. Conversation history is maintained automatically.
    Provider: gemini/openai/anthropic/huggingface/perplexity/all/auto"""
    if provider == 'all': return deep_research(message)
    if provider == 'auto':
        for p in ['gemini', 'openai', 'anthropic', 'huggingface', 'perplexity']:
            if KEYS.get(p): provider = p; break
        if provider == 'auto': print('❌ No keys! Run Cell 1.'); return
    keys = KEYS.get(provider, [])
    if not keys: print(f'❌ No keys for {provider}'); return
    for key, label in keys:
        try:
            s = time.time()
            r = _call(provider, key, message)
            elapsed = int((time.time() - s) * 1000)
            # ═══ PERSIST: Add both user message and response to history ═══
            conversation_history.append({'role': 'user', 'content': message})
            conversation_history.append({'role': 'assistant', 'content': r})
            _trim_history()
            ctx = f'{len(conversation_history)//2} turns'
            print(f'🐝 HeadyBrain [{provider}/{label}] ({elapsed}ms, {ctx}):\n\n{r}')
            return r
        except Exception as e:
            print(f'  ⚠ {label}: {str(e)[:80]}, trying next...')
    print(f'❌ All keys failed for {provider}')


def deep_research(message):
    """Query ALL providers simultaneously. Each gets the full conversation history."""
    print('🔬 Deep Research: querying ALL providers...\n')
    tasks = [(p, ks[0][0], ks[0][1]) for p, ks in KEYS.items() if ks]
    results = []; start = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as pool:
        futs = {pool.submit(_call, p, k, message): (p, l) for p, k, l in tasks}
        for f in concurrent.futures.as_completed(futs):
            p, l = futs[f]
            try:
                r = f.result(timeout=120)
                print(f'✅ {p}/{l}: {len(r)} chars')
                results.append((p, l, r))
            except Exception as e: print(f'❌ {p}/{l}: {str(e)[:60]}')
    # Persist the user message + best result in history
    conversation_history.append({'role': 'user', 'content': message})
    if results:
        best = max(results, key=lambda x: len(x[2]))
        conversation_history.append({'role': 'assistant', 'content': best[2]})
    _trim_history()
    print(f'\n═══ {len(results)}/{len(tasks)} ({int((time.time()-start)*1000)}ms) ═══\n')
    for p, l, r in results:
        print(f'── {p.upper()}/{l} ({len(r)} chars) ──\n{r[:3000]}')
        if len(r) > 3000: print(f'... [{len(r)-3000} more]')
        print()
    return results


print(f'💬 Chat ready! Session: {session_id} | History: {len(conversation_history)} msgs')
print('  chat(\"your message\")              → auto (continues conversation)')
print('  chat(\"your message\", \"openai\")     → GPT-4o')
print('  chat(\"your message\", \"anthropic\")  → Claude')
print('  chat(\"your message\", \"all\")        → ALL providers at once')
print('  new_chat()                         → start fresh conversation')
print('  show_history()                     → view conversation so far')
print('  save_chat(\"name\") / load_chat(\"name\") → persist across restarts')

In [ ]:
# ═══ Cell 3: 💬 CHAT ═══
chat("What should we prioritize to get Heady to enterprise grade?")

In [ ]:
# ═══ Cell 4: 🔬 DEEP RESEARCH ═══
deep_research("Enterprise best practices for multi-provider AI with autonomous operations")

In [ ]:
# ═══ Cell 5: 🔄 Interactive Chat (with persistent history) ═══
print('🐝 HeadyBrain — type and talk! Conversation history is maintained.')
print('Prefix: @openai @claude @gemini @hf @all')
print('Commands: /new (fresh chat) /save [name] /load [name] /history /list /quit\n')
while True:
    try:
        msg = input('You: ')
        if msg.lower() in ('quit', 'exit', 'q', '/quit'): break
        if not msg.strip(): continue
        # Session management commands
        if msg.strip() == '/new':
            new_chat(); continue
        if msg.strip() == '/history':
            show_history(); continue
        if msg.strip() == '/list':
            list_chats(); continue
        if msg.strip().startswith('/save'):
            name = msg.strip().split(' ', 1)[1] if ' ' in msg.strip() else 'auto'
            save_chat(name); continue
        if msg.strip().startswith('/load'):
            name = msg.strip().split(' ', 1)[1] if ' ' in msg.strip() else 'auto'
            load_chat(name); continue
        p = 'auto'
        for pfx, prov in [('@openai ', 'openai'), ('@claude ', 'anthropic'), ('@gemini ', 'gemini'), ('@hf ', 'huggingface'), ('@all ', 'all')]:
            if msg.startswith(pfx): p, msg = prov, msg[len(pfx):]; break
        print(); chat(msg, p); print()
    except KeyboardInterrupt:
        print('\n🛑 Done.'); break